# Generate Audio from Audio_Narration Files
This notebook generates WAV audio files for each narration part in the `Audio_Narration` folder using the **Google Gemini TTS API** (`gemini-2.5-flash-preview-tts`).

## 1. Install Dependencies & Import Libraries

In [1]:
import mimetypes
import os
import re
import struct
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types


## 2. Configure Gemini Client

In [2]:
# Load .env file from the same folder as this notebook (Explainer001/.env)
_env_path = Path().resolve() / ".env"
load_dotenv(dotenv_path=_env_path, override=False)

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise EnvironmentError(
        f"GEMINI_API_KEY not found. Checked env var and {_env_path}"
    )

client = genai.Client(api_key=GEMINI_API_KEY)

TTS_MODEL = "gemini-2.5-flash-preview-tts"

VOICE_NAME = "Charon"  # Sample config voice

VOICE_ROLE_PROMPT = (
    "Act as a professional voice-over artist named 'Varun'. "
    "Persona: 27-year-old male. "
    "Style: High-energy Senior Software Engineer (like a tech YouTuber). "
    "Your tone is clear, confident, and uses a hook-driven delivery. "
    "Voice Profile: Mid-high masculine pitch (250Hz), zero nasal resonance, stable professional delivery. "
)

generate_content_config = types.GenerateContentConfig(
    temperature=1,
    response_modalities=["audio"],
    speech_config=types.SpeechConfig(
        voice_config=types.VoiceConfig(
            prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=VOICE_NAME)
        )
    ),
)

print(f"Client initialised. Model: {TTS_MODEL} | Voice: {VOICE_NAME}")
print(f"API key loaded from: {'env var' if not _env_path.exists() else _env_path}")

Client initialised. Model: gemini-2.5-flash-preview-tts | Voice: Charon
API key loaded from: /Users/abhisheksaxena/Tutorials/prompt-shot/.env


## 3. Load SSML Files from Audio_Narration Folder

In [3]:
# Resolve Audio_Narration folder relative to this notebook
NOTEBOOK_DIR = Path().resolve()
AUDIO_NARRATION_DIR = NOTEBOOK_DIR / "voiceover"

if not AUDIO_NARRATION_DIR.exists():
    raise FileNotFoundError(f"Audio_Narration folder not found at: {AUDIO_NARRATION_DIR}")

# Collect all XML narration part files, sorted by name
narration_files = sorted(AUDIO_NARRATION_DIR.glob("*.xml"))

if not narration_files:
    raise FileNotFoundError("No .xml files found in Audio_Narration folder.")

print(f"Found {len(narration_files)} narration file(s):\n")
for f in narration_files:
    content = f.read_text(encoding="utf-8")
    print(f"{'='*60}")
    print(f"File: {f.name}")
    print(f"{'='*60}")
    print(content[:400], "..." if len(content) > 400 else "")
    print()

Found 1 narration file(s):

File: 06_generate_image.xml
<?xml version="1.0" encoding="UTF 8"?>
<speak>

 <p>
 <s>Cells <say as interpret as="cardinal">6</say as> and
 <say as interpret as="cardinal">7</say as> run single image generation
 at <say as interpret as="cardinal">1024</say as> by
 <say as interpret as="cardinal">1024</say as> pixels 
 the resolution the FLUX architecture was trained at 
 and demonstrate the most important hyperparameter in di ...



## 4. Define WAV Conversion Utilities

In [4]:
def parse_audio_mime_type(mime_type: str) -> dict:
    """Extract bits_per_sample and rate from an audio MIME type string.
    e.g. 'audio/L16;rate=24000' -> {'bits_per_sample': 16, 'rate': 24000}
    """
    bits_per_sample = 16
    rate = 24000
    for param in mime_type.split(";"):
        param = param.strip()
        if param.lower().startswith("rate="):
            try:
                rate = int(param.split("=", 1)[1])
            except (ValueError, IndexError):
                pass
        elif param.startswith("audio/L"):
            try:
                bits_per_sample = int(param.split("L", 1)[1])
            except (ValueError, IndexError):
                pass
    return {"bits_per_sample": bits_per_sample, "rate": rate}


def convert_to_wav(audio_data: bytes, mime_type: str) -> bytes:
    """Prepend a valid WAV file header to raw PCM audio bytes.
    Reference: http://soundfile.sapp.org/doc/WaveFormat/
    """
    params = parse_audio_mime_type(mime_type)
    bits_per_sample = params["bits_per_sample"]
    sample_rate = params["rate"]
    num_channels = 1
    data_size = len(audio_data)
    bytes_per_sample = bits_per_sample // 8
    block_align = num_channels * bytes_per_sample
    byte_rate = sample_rate * block_align
    chunk_size = 36 + data_size

    header = struct.pack(
        "<4sI4s4sIHHIIHH4sI",
        b"RIFF", chunk_size, b"WAVE",
        b"fmt ", 16, 1, num_channels,
        sample_rate, byte_rate, block_align, bits_per_sample,
        b"data", data_size,
    )
    return header + audio_data


print("WAV conversion utilities ready.")

WAV conversion utilities ready.


## 5. Define Audio Generation Function

In [5]:
def generate_audio_for_file(
    ssml_content: str,
    output_path: Path,
    role_prompt: str = VOICE_ROLE_PROMPT,
) -> Path | None:
    """Stream Gemini TTS audio for the given SSML content and save a single WAV file.

    Bug fix: previously each streaming chunk was written as a separate file
    with its own WAV header, which caused narration to repeat when files were
    concatenated during playback. Now all PCM chunks are accumulated and
    concatenated first, then wrapped in exactly ONE WAV header.

    Args:
        ssml_content: Full SSML/XML string from an Audio_Narration part file.
        output_path:  Base output path (without extension).
        role_prompt:  Narrator role instructions prepended to the SSML.

    Returns:
        Path of the audio file written, or None if no audio was received.
    """
    prompt_text = f"{role_prompt}\n{ssml_content}"

    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=prompt_text)],
        )
    ]

    pcm_chunks: list[bytes] = []
    mime_type: str | None = None

    for chunk in client.models.generate_content_stream(
        model=TTS_MODEL,
        contents=contents,
        config=generate_content_config,
    ):
        if chunk.parts is None:
            continue

        part = chunk.parts[0]
        if part.inline_data and part.inline_data.data:
            # Capture mime_type from first chunk; accumulate raw PCM bytes
            if mime_type is None:
                mime_type = part.inline_data.mime_type
            pcm_chunks.append(part.inline_data.data)
        elif hasattr(chunk, "text") and chunk.text:
            print(f"  API message: {chunk.text}")

    if not pcm_chunks:
        print("  WARNING: No audio data received.")
        return None

    # Concatenate all PCM fragments, then wrap with ONE WAV header
    raw_pcm = b"".join(pcm_chunks)
    detected_ext = mimetypes.guess_extension(mime_type) if mime_type else None
    if detected_ext is None:
        detected_ext = ".wav"
        audio_data = convert_to_wav(raw_pcm, mime_type or "audio/L16;rate=24000")
    else:
        audio_data = raw_pcm

    file_path = output_path.with_suffix(detected_ext)
    file_path.write_bytes(audio_data)
    print(f"  Saved: {file_path.name}  ({len(audio_data) / 1024:.1f} KB)")
    return file_path


print("generate_audio_for_file() defined and ready.")


generate_audio_for_file() defined and ready.


## 6. Batch Process All Files and Save Output

In [6]:
import time

# Output folder for generated audio files
OUTPUT_DIR = NOTEBOOK_DIR / "Audio"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_generated: list[Path] = []
failed: list[str] = []

for i, narration_file in enumerate(narration_files):
    print(f"\nProcessing: {narration_file.name}")
    ssml_content = narration_file.read_text(encoding="utf-8")
    output_base = OUTPUT_DIR / narration_file.stem

    try:
        file_path = generate_audio_for_file(ssml_content, output_base)
        if file_path:
            all_generated.append(file_path)
    except Exception as e:
        print(f"  ERROR processing {narration_file.name}: {e}")
        failed.append(narration_file.name)

    # Wait 3 minutes between calls to avoid rate limiting (skip after last file)
    if i < len(narration_files) - 1:
        print(f"  Waiting 3 minutes before next file...")
        time.sleep(180)

# Summary
print("\n" + "="*60)
print("BATCH COMPLETE")
print("="*60)
print(f"Total audio files generated : {len(all_generated)}")
print(f"Failed parts                : {len(failed)}")
if failed:
    print(f"Failed files: {failed}")
print("\nGenerated files:")
for f in all_generated:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}  ({size_kb:.1f} KB)")



Processing: 06_generate_image.xml
  Saved: 06_generate_image.wav  (5871.2 KB)

BATCH COMPLETE
Total audio files generated : 1
Failed parts                : 0

Generated files:
  06_generate_image.wav  (5871.2 KB)
